# 05 Baseline Tweedie GLM

**Purpose:** first model for all three lines, with the validation and checks fixed in DECISIONS.md
(Oct 18 modelling plan): train on 2021–2023, validate on 2024, Tweedie deviance (power 1.5).
Then refit on 2021–2024 and write the first submission file.

Target and weight come from `build_model_target` (2025 level; weight = exposure × completeness).
Features come from `build_features` (excluded columns and reasons are listed in `src/features.py`).

In [ ]:
import sys, time
sys.path.append("..")
from pathlib import Path
import numpy as np
import pandas as pd
from src.target import DATA_DIR, LINES, load_policies, load_claims, build_model_target
from src.features import build_features
from src.glm import BandedTweedieGLM
from src.evaluate import evaluate

claims = load_claims()
SEGMENTS = {  # columns to check A/E by on validation
    "motor": ["driver_age", "vehicle_value_group", "region", "new_business", "cover_type", "telematics_opt_in"],
    "buildings": ["policyholder_age", "sum_insured", "region", "flood_zone", "soil_type", "dry_summer_index"],
    "contents": ["policyholder_age", "sum_insured", "region", "tenure", "security_level", "rainfall_mm"],
}

## 1. Build the modelling table for each line
**Purpose:** join the 2025-level target and weight to the features; check row counts and missing values.

In [ ]:
data = {}
for line in LINES:
    p = load_policies(line, "train")
    mt, comp, loading = build_model_target(claims, p, line)
    F = build_features(line, "train", policies=p)
    df = mt.merge(F, on="row_id", how="left", validate="one_to_one")
    assert len(df) == len(p)
    data[line] = {"df": df, "loading": loading}
    feats = [c for c in F.columns if c != "row_id"]
    miss = df[feats].isna().mean()
    print(f"{line}: {len(df):,} rows, {len(feats)} features, loading {loading:.1f}; "
          f"features with missing values: {miss[miss > 0].round(3).to_dict()}")

## 2. Validation: train 2021–2023, predict 2024
**Purpose:** the main model check. Predicted cost per row = predicted rate × weight, compared with the
actual 2025-level cost of 2024 policies. Segment flags show where A/E is outside 0.9–1.1.

In [ ]:
results, valid_rows = {}, []
for line in LINES:
    df = data[line]["df"]
    feats = [c for c in df.columns if c not in ("row_id", "year", "exposure_clean", "target", "weight")]
    tr, va = df[df.year <= 2023], df[df.year == 2024]
    t0 = time.time()
    glm = BandedTweedieGLM().fit(tr[feats], tr.target / tr.weight, tr.weight)
    pred = glm.predict_rate(va[feats]) * va.weight.to_numpy()
    print(f"\n[{line}] fitted in {time.time() - t0:.0f}s")
    results[line] = evaluate(va.target, pred, va.weight, segments=va[SEGMENTS[line]], name=f"GLM {line} 2024")
    valid_rows.append(pd.DataFrame({"actual": va.target.to_numpy(), "pred": pred, "weight": va.weight.to_numpy()}))

In [ ]:
# All lines together (closest to how Kaggle scores one file)
v = pd.concat(valid_rows, ignore_index=True)
overall = evaluate(v.actual, v.pred, v.weight, name="GLM all lines 2024", show=False)
summary = pd.DataFrame({k: {"D2": r["D2"], "A/E": r["A/E"], "top/bottom": r["top/bottom"]}
                        for k, r in {**results, "all lines": overall}.items()}).T
print(summary.round(4))

## 3. Refit on 2021–2024 and predict 2025
**Purpose:** use all years for the submission. Row prediction = (predicted rate + large-loss loading) × test exposure.

In [ ]:
subs, models = [], {}
for line in LINES:
    df = data[line]["df"]
    feats = [c for c in df.columns if c not in ("row_id", "year", "exposure_clean", "target", "weight")]
    glm = BandedTweedieGLM().fit(df[feats], df.target / df.weight, df.weight)
    models[line] = glm
    pt = load_policies(line, "test")
    Ft = build_features(line, "test", policies=pt)
    rate = glm.predict_rate(Ft[feats])
    pred = np.maximum((rate + data[line]["loading"]) * pt.exposure_clean.to_numpy(), 0)
    subs.append(pd.DataFrame({"row_id": Ft.row_id, "prediction": pred}))
    exp_level = df.target.sum() / df.weight.sum() + data[line]["loading"]
    print(f"{line}: mean predicted rate {rate.mean() + data[line]['loading']:.1f} "
          f"(training level incl. loading {exp_level:.1f}); {len(pt):,} test rows")

In [ ]:
sub = pd.concat(subs, ignore_index=True)
sample = pd.read_parquet(DATA_DIR / "sample_submission.parquet")
assert len(sub) == len(sample) and set(sub.row_id) == set(sample.row_id), "row_ids must match sample_submission"
assert sub.prediction.notna().all() and (sub.prediction >= 0).all()
out = Path("../submissions"); out.mkdir(exist_ok=True)
sub.to_csv(out / "glm_baseline.csv", index=False)
print(f"wrote {len(sub):,} rows to submissions/glm_baseline.csv")
print(sub.prediction.describe().round(1))

## 4. Relativities (for review)
**Purpose:** check that the main effects make actuarial sense (sign and size), e.g. young drivers higher.

In [ ]:
rel = models["motor"].relativities()
for f in ["driver_age", "ncb_years", "vehicle_value_group", "credit_band", "region"]:
    print(rel[rel.feature == f].round(3).to_string(index=False), "\n")